# اكتشاف المعادن: مكتبة USGS Splib07 + صور الأقمار الصناعية
خطوات الدفتر: (1) تثبيت الحزمة (2) تنزيل مكتبة USGS وبناؤها (3) تحميل صورة قمر (4) المطابقة وعرض الخرائط (5) حفظ النتائج.

**مهم:** الصورة يجب أن تكون **انعكاسًا سطحيًا** (Sentinel-2 L2A مثلًا) لا قيم DN أو TOA.

In [ ]:
!pip -q install "git+https://github.com/yamenetoo/RremoteSensing.git#subdirectory=USGS" rasterio matplotlib tqdm

## 1) المكتبة الطيفية (تنزيل مرة واحدة وحفظها في Drive)

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

WORK = Path("/content/drive/MyDrive/splib07")   # يُحفظ هنا ولا يُنزَّل ثانية
WORK.mkdir(parents=True, exist_ok=True)
CACHE = WORK / "splib07_vnir_swir.npz"

In [ ]:
from usgs_splib07 import SpectralLibrary, build_library, download_splib07

if CACHE.exists():
    lib = SpectralLibrary.load(CACHE)
else:
    root = download_splib07(WORK)        # من USGS ScienceBase (قد يستغرق دقائق)
    lib = build_library(root, library="splib07b", chapters="MSV",   # M معادن، S تربة، V نباتات
                        kinds=("AREF",), start=350, stop=2500, step=5)
    lib.save(CACHE)
print(lib)
lib.meta["material"].value_counts().head(15)

## 2) الصورة
اختر **واحدًا** من الخيارات:
- **A** رفع ملف GeoTIFF لديك.
- **B** تصدير Sentinel-2 L2A من Google Earth Engine.
- **C** مشهد اصطناعي للتجربة فقط (لا بيانات حقيقية).

In [ ]:
SOURCE = "gee"   # "gee" = download Sentinel-2 from Earth Engine | "upload" = your own GeoTIFF | "synthetic" = fake demo scene

if SOURCE == "upload":
    # ---------- A) ارفع ملف GeoTIFF ----------
    from google.colab import files
    up = files.upload()
    IMG_PATH = next(iter(up))
    # اضبط حسب بياناتك:
    SENSOR = "sentinel2a"                                   # أو landsat8 / aster
    BANDS  = ["B2","B3","B4","B5","B6","B7","B8","B8A","B11","B12"]  # ترتيب النطاقات في الملف
    SCALE, OFFSET = 1e-4, 0.0                               # Sentinel-2 L2A. لاندسات C2 L2: 2.75e-5 و -0.2

In [ ]:
if SOURCE == "gee":
    # ---------- B) Google Earth Engine -> download straight into Colab (no Drive), tile by tile ----------
    import ee, requests, numpy as np, rasterio
    from rasterio.merge import merge

    ee.Authenticate(); ee.Initialize(project="ee-almohamadmohamad678")

    WEST, SOUTH, EAST, NORTH = 34.9, 29.4, 35.1, 29.6
    START, END, MAX_CLOUD = "2024-01-01", "2024-12-31", 10
    BANDS = ["B2","B3","B4","B5","B6","B7","B8","B8A","B11","B12"]
    TILE = 0.05          # degrees per download tile (about 5 km, ~280x280 pixels): keeps each request small

    region = ee.Geometry.Rectangle([WEST, SOUTH, EAST, NORTH])
    col = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
           .filterBounds(region).filterDate(START, END)
           .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", MAX_CLOUD)))
    print("scenes:", col.size().getInfo())

    crs = col.first().select("B11").projection().crs().getInfo()          # UTM zone of the scenes
    # no clip(): each tile's region does the cropping. Masked pixels -> -9999
    img = col.select(BANDS).median().unmask(-9999).toInt16().reproject(crs=crs, scale=20)

    def download_tile(w, s, e, n, path):
        box = ee.Geometry.Rectangle([w, s, e, n], proj="EPSG:4326", geodesic=False)
        url = img.getDownloadURL({"region": box, "scale": 20, "crs": crs, "format": "GEO_TIFF"})
        r = requests.get(url, timeout=600)
        if r.status_code != 200:
            raise RuntimeError(f"HTTP {r.status_code}: {r.text[:500]}")   # shows Earth Engine's real reason
        open(path, "wb").write(r.content)

    xs = np.arange(WEST, EAST - 1e-9, TILE); ys = np.arange(SOUTH, NORTH - 1e-9, TILE)
    paths = []
    for i, s in enumerate(ys):
        for j, w in enumerate(xs):
            p = f"/content/tile_{i}_{j}.tif"
            download_tile(w, s, min(w + TILE, EAST), min(s + TILE, NORTH), p)
            paths.append(p)
    print(len(paths), "tiles downloaded")

    # mosaic the tiles into one GeoTIFF
    srcs = [rasterio.open(p) for p in paths]
    mosaic, transform = merge(srcs, nodata=-9999)
    meta = srcs[0].meta.copy()
    meta.update(height=mosaic.shape[1], width=mosaic.shape[2], transform=transform, nodata=-9999)
    IMG_PATH = "/content/s2_scene.tif"
    with rasterio.open(IMG_PATH, "w", **meta) as dst:
        dst.write(mosaic)
    for s_ in srcs: s_.close()
    print(f"saved {IMG_PATH}: {mosaic.shape[2]} x {mosaic.shape[1]} px, {mosaic.shape[0]} bands, crs={crs}")
    SENSOR, SCALE, OFFSET = "sentinel2a", 1e-4, 0.0

In [ ]:
if SOURCE == "synthetic":      # only for testing: replaces lib and image with FAKE data
    # ---------- C) مشهد اصطناعي للتجربة ----------
    from usgs_splib07.synthetic import demo_library, synthetic_scene
    demo = demo_library(step=1.0)
    image, truth, labels = synthetic_scene(demo, "sentinel2a", noise=0.005, seed=1)
    lib = demo                 # يستبدل المكتبة الحقيقية في هذا الخيار فقط

## 3) قراءة الصورة (تخطَّ هذه الخلية في الخيار C)

In [ ]:
if SOURCE != "synthetic":
    from usgs_splib07 import Image
    image = Image.read(IMG_PATH, sensor=SENSOR, band_names=BANDS, scale=SCALE, offset=OFFSET, nodata=-9999)
    print(image.n_bands, "bands,", image.rows, "x", image.cols)
    # صورة فائقة الطيف (EnMAP/PRISMA/EMIT محوّلة إلى GeoTIFF)؟ إن كانت الأطوال في الوسوم يكفي:
    # image = Image.read("scene.tif", scale=1e-4)    أو مرّرها: wavelengths=[...], fwhm=10

print(lib)   # must show thousands of spectra and many materials, not n=8 'synthetic'
assert "synthetic" not in lib.history, "lib is the FAKE demo library: restart the runtime and use SOURCE != 'synthetic'"

## 4) المطابقة

In [ ]:
import math
import numpy as np, matplotlib.pyplot as plt
from usgs_splib07 import match_image

# keep_scores=True keeps the spectral angle to EVERY material (needed for the per-material maps)
res = match_image(image, lib,
                  group_by="material",     # best sample of each material
                  max_angle_deg=10.0,      # largest accepted spectral angle (lower = stricter)
                  mask_vegetation=0.4,     # ignore green pixels (NDVI > 0.4)
                  keep_scores=True)
display(res.counts().head(15))

# ------------------------------- SETTINGS ------------------------------------
TEMPERATURE = 2.0     # degrees. Small = sharp (winner takes all), large = probability shared by similar materials
NCOLS, PER_PAGE = 4, 12   # panels per row / per figure (all materials are shown, page after page)
ONLY_PRESENT = False  # False: ONE image per material in the library (= len(res.labels)); True: only materials that won somewhere
MIN_P = 0.10          # probabilities below this stay transparent so the satellite image shows through
# ------------------------------------------------------------------------------

labels = np.array(res.labels, dtype=object)
n_best = np.bincount(res.class_map[res.class_map >= 0], minlength=len(labels))
mats = [k for k in np.argsort(-n_best, kind="stable") if (n_best[k] > 0 or not ONLY_PRESENT)]
print(f"{len(mats)} materials -> {len(mats)} images")

# true-colour satellite image used as the base of every panel
bands = [image.band_names.index(n) for n in ("B4", "B3", "B2")] if image.band_names and "B4" in image.band_names else [0, 1, 2]
rgb = np.nan_to_num(image.data[bands]).astype("float32")
lo, hi = np.percentile(rgb[rgb > 0], (2, 98)) if (rgb > 0).any() else (0, 1)
rgb = np.clip(np.moveaxis((rgb - lo) / max(hi - lo, 1e-6), 0, -1), 0, 1)

# map frame: use cartopy with the image's own UTM projection when the image is georeferenced
proj, extent = None, None
if image.transform is not None and image.crs is not None:
    t = image.transform
    extent = (t.c, t.c + t.a * image.cols, t.f + t.e * image.rows, t.f)       # left, right, bottom, top
    try:
        import cartopy.crs as ccrs
        epsg = image.crs.to_epsg()
        if 32601 <= epsg <= 32660:
            proj = ccrs.UTM(epsg - 32600)
        elif 32701 <= epsg <= 32760:
            proj = ccrs.UTM(epsg - 32700, southern_hemisphere=True)
    except Exception:
        proj = None


def draw(ax, p, title):
    if proj is not None:
        ax.imshow(rgb, origin="upper", extent=extent, transform=proj, interpolation="nearest")
        im = ax.imshow(np.ma.masked_less(p, MIN_P), origin="upper", extent=extent, transform=proj,
                       cmap="turbo", vmin=0, vmax=1, alpha=0.75, interpolation="nearest")
        ax.set_extent(extent, crs=proj)
        gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="white", alpha=0.5, linestyle="--")
        gl.top_labels = gl.right_labels = False
        gl.xlabel_style = gl.ylabel_style = {"size": 6}
    else:
        ax.imshow(rgb)
        im = ax.imshow(np.ma.masked_less(p, MIN_P), cmap="turbo", vmin=0, vmax=1, alpha=0.75, interpolation="nearest")
        ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=9)
    return im


n_valid = max(int((res.class_map >= 0).sum()), 1)
for page in range(math.ceil(len(mats) / PER_PAGE)):
    ks = mats[page * PER_PAGE:(page + 1) * PER_PAGE]
    P = res.probability(TEMPERATURE, indices=ks)                      # (len(ks), rows, cols), 0..1
    nrows = math.ceil(len(ks) / NCOLS)
    fig = plt.figure(figsize=(4.4 * NCOLS, 4.4 * nrows))
    for n, (k, p) in enumerate(zip(ks, P)):
        ax = fig.add_subplot(nrows, NCOLS, n + 1, projection=proj)
        im = draw(ax, p, f"{labels[k]}\nbest in {n_best[k]} px ({n_best[k] / n_valid:.1%}) | max P = {np.nanmax(p):.2f}")
    fig.subplots_adjust(right=0.9, wspace=0.25, hspace=0.35)
    cax = fig.add_axes([0.92, 0.25, 0.012, 0.5])
    fig.colorbar(im, cax=cax, label="probability")
    fig.suptitle(f"Material probability maps  (page {page + 1}/{math.ceil(len(mats) / PER_PAGE)}, "
                 f"materials {page * PER_PAGE + 1}-{page * PER_PAGE + len(ks)} of {len(mats)})", fontsize=12)
    plt.show()


`angle_deg` صغيرة = تشابه جيد. `margin_deg` (الفرق بين أفضل وثاني أفضل مادة) صغير = نتيجة ملتبسة، فلا تثق بها.

In [ ]:
# نتيجة موثوقة فقط: زاوية صغيرة وهامش واضح
good = (res.angle_deg < 6) & (res.margin_deg > 1.0)
print(f"{good.mean():.1%} of pixels are confident matches")
plt.figure(figsize=(6,5)); plt.imshow(np.ma.masked_where(~good | (res.class_map < 0), res.class_map), cmap="tab20", interpolation="nearest")
plt.title("confident matches only"); plt.axis("off"); plt.show()

## 5) حفظ النتائج

In [ ]:
# GeoTIFF من 3 نطاقات (class, angle, margin) + ملف legend.csv. يحتاج صورة بإحداثيات (الخيار A/B).
out = res.to_geotiff("/content/drive/MyDrive/splib07/match.tif")
print(out)